# Reasoning Paradigm Study — Analysis

Compares four agent reasoning paradigms (**ReAct, Plan-and-Solve, ReWOO, Reflection**) plus two baselines (**Direct LLM, RAG Baseline**) on the Yelp RAG-QA task, across two DeepSeek-V4 models (Flash, Pro) and two thinking modes.

Reads `results/paradigm_study.json` (320 rows, LLM-as-judge scored on 4 quality dimensions, 0–2 each → quality /8). Efficiency (latency / cost / tokens) comes from raw run metrics.

**Headline finding:** answer quality is *saturated* (≈8/8) for every retrieval-grounded paradigm; only Direct LLM (no retrieval) fails. Paradigm choice, thinking mode, and model size move cost/latency dramatically but **not** quality — so the paradigm decision is an *efficiency* decision, not a quality one.

## Cell 1 — Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

%matplotlib inline
import pandas as pd
from yelp_rag_agent.evaluation import paradigm_figures as pf

df = pf.load_study()
print('rows:', len(df), '| models:', df['model'].unique().tolist())
print('quality /8 range:', df['quality8'].min(), '-', df['quality8'].max())
df.head(3)

## Cell 2 — Quality ceiling
Every retrieval paradigm lands at ≈8/8; only Direct LLM (no retrieval) collapses. Quality does not separate the paradigms.

In [ ]:
pf.fig_quality_ceiling(df);

## Cell 3 — Quality vs latency (the money chart)
Quality is a flat band near the ceiling while latency spans 30×+. ○ Flash, △ Pro; black edge = thinking on; colour = paradigm.

In [ ]:
pf.fig_quality_vs_latency(df);

## Cell 4 — The thinking tax
Thinking mode adds ~3–4× latency (and ~2× cost) for no measurable quality gain.

In [ ]:
pf.fig_latency_thinking_tax(df);

## Cell 5 — Token cost breakdown
ReAct re-sends the growing message history each turn (high input tokens); thinking mode adds a large reasoning-token tail.

In [ ]:
pf.fig_token_breakdown(df);

## Cell 6 — Summary tables

In [ ]:
main = pf.table_main(df)
main

In [ ]:
tax = pf.table_thinking_tax(df)
tax

## Conclusions

1. **Retrieval, not reasoning structure, drives quality.** Direct LLM ≈ 0.2/8; every retrieval-grounded system ≈ 7.8–8.0/8.
2. **Quality is saturated across paradigms.** ReAct / Plan-and-Solve / ReWOO / Reflection are statistically indistinguishable on quality — the paradigm tradeoff is cost & latency.
3. **The thinking tax is real and unrewarded.** Thinking mode costs ~3–4× latency and ~2× tokens for ≈ 0 quality gain on this task (Pro: +0.00/8).
4. **Pro is not worth it here.** ≈10× the cost and ~3× the latency of Flash for the same quality.
5. **Practical recommendation:** Plan-and-Solve or ReWOO on Flash with thinking off — ceiling-quality answers at the lowest cost/latency. Reflection and thinking add overhead without payoff once quality is saturated.

**Caveats:** LLM-as-judge (DeepSeek-V4 judging V4 — self-preference bias, uniform across paradigms so relative comparisons hold); the 0–2 rubric is too coarse to separate near-ceiling answers; groundedness assessed from internal consistency since retrieved chunks were not stored.